# Fine-tune Qwen3-Embedding-4B trên Kaggle — QLoRA, tối đa 12 giờ

**Đây là notebook fine-tune Qwen bạn cần chạy.** GPU T4, Internet ON, Save Version
→ Save & Run All. Dùng đúng model `Qwen/Qwen3-Embedding-4B`, không dùng Qwen chat.
Một GPU được sử dụng; hai T4 không tạo thành một bộ nhớ chung.

Theo hướng dẫn Qwen: SWIFT, `task_type=embedding`, `model_type=qwen3_emb`, InfoNCE,
dataset `messages` / `positive_messages` / `negative_messages`. Chuyển từ ví dụ
full train nhiều GPU sang **QLoRA NF4 4-bit**, rank 8, batch 1, một explicit negative
để phù hợp T4; đây là cấu hình khởi đầu cần đo, chưa được cam kết GPU nào cũng đủ RAM.
Không bật BF16 hoặc FlashAttention 2 trên T4.

**Ngân sách tính cả cài đặt và tải dữ liệu/model:** dừng train ở giờ 10; hard stop
worker train ở 10h30; dừng đánh giá ở giờ 11; giữ giờ cuối cho đóng gói và Kaggle
lưu Output. Callback lưu mỗi 15 phút hoặc 100 optimizer steps; checkpoint có cả
optimizer/RNG để tiếp tục trong phiên sau. Giới hạn này giảm rủi ro mất kết quả,
không thể bảo đảm chống việc nền tảng tắt đột ngột hoặc tải mạng quá chậm.

Output chính là **adapter LoRA + base revision đã pin**, không merge 8GB weights
trong phiên train. Notebook 05 đã hỗ trợ nạp adapter này để chạy GPU endpoint.
Notebook 06 không mở tunnel hay chờ nhập key; Save & Run All sẽ kết thúc.

Tài liệu dùng để triển khai:
- https://github.com/QwenLM/Qwen3-Embedding/blob/main/docs/training/SWIFT.md
- https://github.com/modelscope/ms-swift/tree/v3.12.3
- https://huggingface.co/docs/peft/developer_guides/quantization
- https://huggingface.co/Qwen/Qwen3-Embedding-4B

In [ ]:
import json, os, sys, time, subprocess, venv, importlib.metadata
from pathlib import Path

# Change these values before Save & Run All.
BASE_MODEL = "Qwen/Qwen3-Embedding-4B"
RUN_NAME = "qwen3-4b-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime())
MAX_LENGTH = 512  # reduce to 256 on OOM; start a new run if changing recipe
LEARNING_RATE = 2e-5
MAX_STEPS = 500  # upper cap, may stop earlier at the 10h deadline
PAIR_LIMIT = 6000
PREPARED_DATA = ""  # optional Research-assist prepared corpus under /kaggle/input
RESUME_CHECKPOINT = ""  # e.g. /kaggle/input/previous-run/RUN/resume
SESSION_ALREADY_ELAPSED_HOURS = 0.0  # enter session age if GPU was started earlier
INSTRUCTION = "Given a research question, retrieve relevant passages that answer the question"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/temp/qwen-hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/qwen-datasets"
os.environ["MODELSCOPE_CACHE"] = "/kaggle/temp/qwen-ms"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["NPROC_PER_NODE"] = "1"

In [ ]:
QWEN_LIBRARY_SOURCE = '"""QASPER paragraphs and gold evidence, preserving official paper-level splits."""\n\nimport hashlib\nimport json\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\n    return [\n        json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()\n    ]\n\n\ndef write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(\n        "".join(json.dumps(row, ensure_ascii=False) + "\\n" for row in rows), encoding="utf-8"\n    )\n\n\ndef file_hash(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef normalize(text: str) -> str:\n    return " ".join(unicodedata.normalize("NFC", text).split())\n\n\ndef records(value: Any) -> list[dict[str, Any]]:\n    """HF\'s columnar Sequence and official raw list have the same canonical representation."""\n    if isinstance(value, list):\n        return value\n    if not value:\n        return []\n    lengths = {len(v) for v in value.values()}\n    if len(lengths) != 1:\n        raise ValueError("Malformed columnar sequence")\n    return [{k: v[i] for k, v in value.items()} for i in range(next(iter(lengths)))]\n\n\ndef prepare_qasper(source: Path, output: Path) -> dict[str, Any]:\n    corpus: list[dict[str, Any]] = []\n    queries: dict[str, list[dict[str, Any]]] = {}\n    pairs: list[dict[str, Any]] = []\n    paper_splits: dict[str, str] = {}\n    stats: Counter[str] = Counter()\n    hashes: dict[str, str] = {}\n    for split in ("train", "validation", "test"):\n        path = source / f"{split}.json"\n        hashes[split] = file_hash(path)\n        raw = json.loads(path.read_text(encoding="utf-8"))\n        papers = raw.items() if isinstance(raw, dict) else ((r["id"], r) for r in raw)\n        queries[split] = []\n        for paper_id, paper in papers:\n            paper_id = str(paper_id)\n            if paper_id in paper_splits:\n                raise ValueError(f"Paper leakage or duplicate: {paper_id}")\n            paper_splits[paper_id] = split\n            stats[f"{split}_papers"] += 1\n            lookup: dict[str, list[str]] = {}\n            paragraphs = [("Abstract", paper.get("abstract", ""))]\n            for section in records(paper["full_text"]):\n                paragraphs.extend(\n                    (section.get("section_name", ""), p) for p in section["paragraphs"]\n                )\n            for index, (section_name, text) in enumerate(paragraphs):\n                if not text.strip():\n                    continue\n                cid = paper_id + ":" + str(index)\n                row = {\n                    "id": cid,\n                    "paper_id": paper_id,\n                    "text": text.strip(),\n                    "section": section_name,\n                    "split": split,\n                }\n                corpus.append(row)\n                lookup.setdefault(normalize(text), []).append(cid)\n            for qa in records(paper["qas"]):\n                answers = [a["answer"] for a in records(qa["answers"])]\n                answerable = [a for a in answers if not a.get("unanswerable", False)]\n                gold = list(dict.fromkeys(e for a in answerable for e in a.get("evidence", [])))\n                textual = [e for e in gold if not e.startswith("FLOAT SELECTED")]\n                ids = sorted({cid for e in textual for cid in lookup.get(normalize(e), [])})\n                unmatched = [e for e in textual if normalize(e) not in lookup]\n                # Partially mapped gold would inflate recall: exclude such questions, report why.\n                reason = (\n                    "unanswerable"\n                    if not answerable\n                    else "non_text_evidence"\n                    if len(textual) != len(gold)\n                    else "unmatched_gold"\n                    if unmatched\n                    else "no_gold"\n                    if not ids\n                    else ""\n                )\n                question = {\n                    "id": str(qa["question_id"]),\n                    "question": qa["question"],\n                    "paper_id": paper_id,\n                    "relevant_ids": ids,\n                    "excluded_reason": reason,\n                    "split": split,\n                }\n                queries[split].append(question)\n                stats[f"{split}_{reason or \'eligible\'}"] += 1\n                if split == "train" and not reason:\n                    pairs.extend(\n                        {\n                            "query_id": question["id"],\n                            "question": question["question"],\n                            "positive_id": cid,\n                            "paper_id": paper_id,\n                            "source": "qasper_gold",\n                            "reviewed": True,\n                        }\n                        for cid in ids\n                    )\n    # Exact duplicate paragraphs across papers can contaminate evaluation too.\n    train_texts = {normalize(c["text"]) for c in corpus if c["split"] == "train"}\n    heldout_texts = {normalize(c["text"]) for c in corpus if c["split"] != "train"}\n    training_text = {c["id"]: normalize(c["text"]) for c in corpus if c["split"] == "train"}\n    original_pairs = len(pairs)\n    pairs = [p for p in pairs if training_text[p["positive_id"]] not in heldout_texts]\n    stats["train_pairs_excluded_duplicate_text"] = original_pairs - len(pairs)\n    stats["train_pairs"] = len(pairs)\n    output.mkdir(parents=True, exist_ok=True)\n    write_jsonl(output / "corpus.jsonl", corpus)\n    write_jsonl(output / "train_pairs.jsonl", pairs)\n    for split, rows in queries.items():\n        write_jsonl(output / f"{split}_queries.jsonl", rows)\n    manifest = {\n        "format_version": 2,\n        "dataset": "allenai/qasper",\n        "language": "en",\n        "source_hashes": hashes,\n        "paper_split": paper_splits,\n        "statistics": dict(stats),\n        "cross_split_duplicate_texts": len(train_texts & heldout_texts),\n        "corpus_sha256": file_hash(output / "corpus.jsonl"),\n        "pairs_sha256": file_hash(output / "train_pairs.jsonl"),\n    }\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return manifest\n\n\ndef load_split(data: Path, split: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\n    corpus = [r for r in read_jsonl(data / "corpus.jsonl") if r["split"] == split]\n    queries = [q for q in read_jsonl(data / f"{split}_queries.jsonl") if not q["excluded_reason"]]\n    ids = {r["id"] for r in corpus}\n    if len(ids) != len(corpus) or not queries or not corpus:\n        raise ValueError("Missing eligible queries/corpus or duplicate corpus IDs")\n    if any(not q["relevant_ids"] or not set(q["relevant_ids"]) <= ids for q in queries):\n        raise ValueError("Gold evidence outside its split")\n    return corpus, queries\n\n\ndef validated_pairs(data: Path, synthetic: Path | None = None) -> list[dict[str, Any]]:\n    corpus = {r["id"]: r for r in read_jsonl(data / "corpus.jsonl")}\n    heldout_texts = {normalize(c["text"]) for c in corpus.values() if c["split"] != "train"}\n    pairs = read_jsonl(data / "train_pairs.jsonl")\n    if synthetic:\n        pairs += read_jsonl(synthetic)\n    result, seen = [], set()\n    for pair in pairs:\n        if pair.get("reviewed") is not True:\n            raise ValueError("Synthetic data must be reviewed by a human before training")\n        positive = corpus.get(pair["positive_id"])\n        if not positive or positive["split"] != "train" or positive["paper_id"] != pair["paper_id"]:\n            raise ValueError("Training evidence leaks held-out papers or has invalid provenance")\n        if normalize(positive["text"]) in heldout_texts:\n            raise ValueError("Training evidence duplicates held-out text")\n        key = (pair["question"], pair["positive_id"])\n        if key not in seen:\n            seen.add(key)\n            result.append({**pair, "positive": positive["text"]})\n    if not result:\n        raise ValueError("No valid training pairs")\n    return result\n\ndef download_qasper(destination, revision=""):\n    from datasets import load_dataset\n    from huggingface_hub import HfApi, hf_hub_download\n\n    destination = Path(destination)\n    api = HfApi()\n    revision = revision or api.dataset_info("allenai/qasper", revision="refs/convert/parquet").sha\n    files = api.list_repo_files("allenai/qasper", repo_type="dataset", revision=revision)\n    paths = {\n        split: [\n            hf_hub_download("allenai/qasper", name, repo_type="dataset", revision=revision)\n            for name in files\n            if name.endswith(".parquet") and f"/{split}/" in name\n        ]\n        for split in ("train", "validation", "test")\n    }\n    if any(not names for names in paths.values()):\n        raise ValueError("Official QASPER parquet conversion is missing a split")\n    dataset = load_dataset("parquet", data_files=paths)\n    destination.mkdir(parents=True, exist_ok=True)\n    for split in paths:\n        rows = {row["id"]: row for row in dataset[split]}\n        (destination / f"{split}.json").write_text(\n            json.dumps(rows, ensure_ascii=False), encoding="utf-8"\n        )\n    metadata = {"dataset": "allenai/qasper", "revision": revision, "license": "CC-BY-4.0"}\n    write_json(destination / "download.json", metadata)\n    return metadata\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n"""SWIFT 3.12.3 QLoRA recipe and bounded Kaggle orchestration, embedded in notebook 06."""\n\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nimport subprocess\nimport time\nfrom pathlib import Path\n\nQWEN_REQUIREMENTS = [\n    "ms-swift==3.12.3",\n    "transformers==4.57.1",\n    "peft==0.17.1",\n    "trl==0.23.1",\n    "datasets==3.6.0",\n    "accelerate==1.11.0",\n    "bitsandbytes==0.48.2",\n    "huggingface-hub==0.36.0",\n    "sentence-transformers==5.1.2",\n    "numpy==2.2.6",\n    "pandas==2.2.3",\n    "pyarrow==20.0.0",\n    "scikit-learn==1.6.1",\n]\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef session_start(clock_path, elapsed_hours=0):\n    """Retain the oldest observed start across cell reruns and kernel restarts."""\n    now = time.time()\n    try:\n        import psutil\n\n        kernel_start = psutil.Process(os.getpid()).create_time()\n    except ImportError:\n        kernel_start = now\n    started = min(kernel_start, now - elapsed_hours * 3600)\n    path = Path(clock_path)\n    if path.exists():\n        started = min(started, json.loads(path.read_text())["started_epoch"])\n    atomic_json(path, {"started_epoch": started})\n    return started\n\n\ndef install_qwen_environment(python, requirements, constraints, deadline):\n    remaining = min(1800, deadline - time.time())\n    if remaining <= 0:\n        raise RuntimeError("No setup budget remaining.")\n    subprocess.run(\n        [\n            str(python),\n            "-m",\n            "pip",\n            "install",\n            "--quiet",\n            "--prefer-binary",\n            "--only-binary=numpy,pandas,pyarrow,scikit-learn",\n            "--upgrade-strategy",\n            "only-if-needed",\n            "-c",\n            str(constraints),\n            *requirements,\n        ],\n        check=True,\n        timeout=remaining,\n    )\n\n\ndef make_triplets(data, tokenizer, instruction, max_length=512, seed=42, limit=6000):\n    """QASPER paper splits, all known gold excluded from lexical negative candidates."""\n    # These helpers are embedded from the repository\'s existing QASPER data preparation.\n\n    corpus, train_queries = load_split(Path(data), "train")\n    positives = validated_pairs(Path(data))\n    excluded_texts = {\n        normalize(item["text"])\n        for item in read_jsonl(Path(data) / "corpus.jsonl")\n        if item["split"] != "train"\n    }\n    by_paper = {}\n    gold = {}\n    for query in train_queries:\n        gold.setdefault(query["question"], set()).update(query["relevant_ids"])\n    for row in corpus:\n        by_paper.setdefault(row["paper_id"], []).append(row)\n    rng = random.Random(seed)\n    rng.shuffle(positives)\n    train, dropped = [], 0\n\n    def fits(text):\n        return len(tokenizer.encode(text, add_special_tokens=False)) + 1 <= max_length\n\n    def row(question, positive, negative):\n        return {\n            "messages": [{"role": "user", "content": f"Instruct: {instruction}\\nQuery:{question}"}],\n            "positive_messages": [[{"role": "user", "content": positive}]],\n            "negative_messages": [[{"role": "user", "content": negative}]],\n        }\n\n    def negative_for(question, paper, forbidden, positive):\n        terms = set(question.casefold().split())\n        candidates = [\n            item\n            for item in by_paper.get(paper, [])\n            if item["id"] not in forbidden\n            and normalize(item["text"]) != normalize(positive)\n            and normalize(item["text"]) not in excluded_texts\n            and fits(item["text"])\n        ]\n        # Lexical candidates are silver negatives: QASPER gold may be incomplete.\n        candidates.sort(\n            key=lambda item: (-len(terms & set(item["text"].casefold().split())), item["id"])\n        )\n        return candidates[0]["text"] if candidates else None\n\n    for item in positives:\n        question, positive = item["question"], item["positive"]\n        query = f"Instruct: {instruction}\\nQuery:{question}"\n        negative = negative_for(\n            question, item["paper_id"], gold.get(question, set()) | {item["positive_id"]}, positive\n        )\n        if not negative or not fits(query) or not fits(positive):\n            dropped += 1\n            continue\n        train.append(row(question, positive, negative))\n        if len(train) >= limit:\n            break\n    validation_corpus, queries = load_split(Path(data), "validation")\n    excluded_texts = set()\n    by_paper.clear()\n    lookup = {item["id"]: item["text"] for item in validation_corpus}\n    for item in validation_corpus:\n        by_paper.setdefault(item["paper_id"], []).append(item)\n    validation = []\n    for query in sorted(queries, key=lambda q: q["id"]):\n        positive = lookup[query["relevant_ids"][0]]\n        negative = negative_for(\n            query["question"], query["paper_id"], set(query["relevant_ids"]), positive\n        )\n        if (\n            negative\n            and fits(positive)\n            and fits(f"Instruct: {instruction}\\nQuery:{query[\'question\']}")\n        ):\n            validation.append(row(query["question"], positive, negative))\n        if len(validation) >= 64:\n            break\n    if len(train) < 16 or len(validation) < 4:\n        raise ValueError("Too few non-truncated triplets. Check data or increase MAX_LENGTH.")\n    return (\n        train,\n        validation,\n        {\n            "train": len(train),\n            "validation": len(validation),\n            "excluded": dropped,\n            "negative_policy": "same-paper lexical silver",\n        },\n    )\n\n\ndef prepare_worker(config):\n    import importlib.metadata\n\n    from huggingface_hub import HfApi, snapshot_download\n    from transformers import AutoTokenizer\n\n\n    run = Path(config["run"])\n    revision = config.get("base_revision")\n    if config.get("resume"):\n        recipe = json.loads((Path(config["resume"]).parent / "resume_recipe.json").read_text())\n        for name in ("base_model", "instruction", "max_length", "learning_rate"):\n            if recipe[name] != config[name]:\n                raise ValueError(f"Resume recipe differs: {name}. Keep original configuration.")\n        if revision and revision != recipe["base_revision"]:\n            raise ValueError("Resume must use the original base revision.")\n        revision = recipe["base_revision"]\n    revision = revision or HfApi().model_info(config["base_model"]).sha\n    config["base_revision"] = revision\n    config["base_path"] = snapshot_download(config["base_model"], revision=revision)\n    data = Path(config.get("prepared_data") or "/kaggle/temp/qwen-qasper")\n    if not config.get("prepared_data"):\n        raw = Path("/kaggle/temp/qwen-qasper-raw")\n        metadata = download_qasper(\n            raw, recipe.get("dataset_revision", "") if config.get("resume") else ""\n        )\n        config["dataset_revision"] = metadata["revision"]\n        manifest = prepare_qasper(raw, data)\n        atomic_json(run / "qasper_download.json", metadata)\n        atomic_json(run / "dataset_manifest.json", manifest)\n    paper_splits, ids = {}, set()\n    for row in read_jsonl(data / "corpus.jsonl"):\n        if row["id"] in ids or paper_splits.get(row["paper_id"], row["split"]) != row["split"]:\n            raise ValueError("Duplicate corpus ID or paper leaking across splits.")\n        ids.add(row["id"])\n        paper_splits[row["paper_id"]] = row["split"]\n    tokenizer = AutoTokenizer.from_pretrained(config["base_path"], trust_remote_code=False)\n    train, validation, counts = make_triplets(\n        data, tokenizer, config["instruction"], config["max_length"], limit=config["pair_limit"]\n    )\n    config["train_data"], config["validation_data"] = (\n        str(run / "train.jsonl"),\n        str(run / "validation.jsonl"),\n    )\n    write_jsonl(Path(config["train_data"]), train)\n    write_jsonl(Path(config["validation_data"]), validation)\n    config["data_sha256"] = {\n        name: hashlib.sha256(Path(config[name]).read_bytes()).hexdigest()\n        for name in ("train_data", "validation_data")\n    }\n    if config.get("resume") and config["data_sha256"] != recipe["data_sha256"]:\n        raise ValueError("Resume data differs from original training run.")\n    corpus, queries = load_split(data, "validation")\n    chosen_queries = sorted(queries, key=lambda item: item["id"])[:16]\n    gold_ids = {cid for query in chosen_queries for cid in query["relevant_ids"]}\n    selected_docs = [item for item in corpus if item["id"] in gold_ids]\n    selected_docs += [item for item in corpus if item["id"] not in gold_ids][\n        : max(0, 256 - len(selected_docs))\n    ]\n    positions = {item["id"]: i for i, item in enumerate(selected_docs)}\n    atomic_json(\n        run / "evaluation_panel.json",\n        {\n            "documents": [item["text"] for item in selected_docs],\n            "questions": [item["question"] for item in chosen_queries],\n            "gold": [[positions[cid] for cid in item["relevant_ids"]] for item in chosen_queries],\n        },\n    )\n    config["packages"] = {\n        name: importlib.metadata.version(name)\n        for name in ("torch", "ms-swift", "transformers", "peft", "trl", "bitsandbytes", "datasets")\n    }\n    atomic_json(run / "config.json", config)\n    atomic_json(run / "triplet_statistics.json", counts)\n\n\ndef swift_arguments(config):\n    """Flags verified against the pinned SWIFT wheel, not a moving latest API."""\n    result = [\n        "--model",\n        config["base_path"],\n        "--model_type",\n        "qwen3_emb",\n        "--task_type",\n        "embedding",\n        "--train_type",\n        "lora",\n        "--tuner_backend",\n        "peft",\n        "--use_hf",\n        "true",\n        "--quant_method",\n        "bnb",\n        "--quant_bits",\n        "4",\n        "--bnb_4bit_compute_dtype",\n        "float16",\n        "--bnb_4bit_quant_type",\n        "nf4",\n        "--bnb_4bit_use_double_quant",\n        "true",\n        "--torch_dtype",\n        "float16",\n        "--fp16",\n        "true",\n        "--bf16",\n        "false",\n        "--attn_impl",\n        "sdpa",\n        "--target_modules",\n        "all-linear",\n        "--lora_rank",\n        "8",\n        "--lora_alpha",\n        "16",\n        "--lora_dropout",\n        "0.05",\n        "--gradient_checkpointing",\n        "true",\n        "--dataset",\n        config["train_data"],\n        "--val_dataset",\n        config["validation_data"],\n        "--split_dataset_ratio",\n        "0",\n        "--max_length",\n        str(config["max_length"]),\n        "--truncation_strategy",\n        "delete",\n        "--loss_type",\n        "infonce",\n        "--label_names",\n        "labels",\n        "--per_device_train_batch_size",\n        "1",\n        "--per_device_eval_batch_size",\n        "1",\n        "--gradient_accumulation_steps",\n        "8",\n        "--learning_rate",\n        str(config["learning_rate"]),\n        "--num_train_epochs",\n        "1",\n        "--max_steps",\n        str(config["max_steps"]),\n        "--warmup_ratio",\n        "0.05",\n        "--eval_strategy",\n        "steps",\n        "--eval_steps",\n        "100",\n        "--save_strategy",\n        "steps",\n        "--save_steps",\n        "100",\n        "--save_total_limit",\n        "2",\n        "--save_only_model",\n        "false",\n        "--load_best_model_at_end",\n        "false",\n        "--metric_for_best_model",\n        "eval_loss",\n        "--greater_is_better",\n        "false",\n        "--logging_steps",\n        "5",\n        "--report_to",\n        "none",\n        "--dataloader_num_workers",\n        "0",\n        "--dataset_num_proc",\n        "1",\n        "--dataloader_drop_last",\n        "true",\n        "--seed",\n        "42",\n        "--data_seed",\n        "42",\n        "--output_dir",\n        config["output_dir"],\n        "--add_version",\n        "false",\n    ]\n    if config.get("resume"):\n        result += ["--resume_from_checkpoint", config["resume"]]\n    return result\n\n\ndef deadline_callback(deadline, progress_path, checkpoint_seconds=900, clock=time.time):\n    from transformers import TrainerCallback\n\n    class DeadlineCallback(TrainerCallback):\n        def __init__(self):\n            self.last_save = clock()\n\n        @staticmethod\n        def signature(model):\n            import torch\n\n            digest, count = hashlib.sha256(), 0\n            for name, parameter in model.named_parameters():\n                if not parameter.requires_grad:\n                    continue\n                value = parameter.detach().float().cpu()\n                if not torch.isfinite(value).all():\n                    raise RuntimeError(\n                        "Non-finite trainable weights; checkpoint is not deployable."\n                    )\n                digest.update(name.encode())\n                digest.update(value.numpy().tobytes())\n                count += value.numel()\n            if not count:\n                raise RuntimeError("No trainable adapter parameters found.")\n            return digest.hexdigest()\n\n        def on_train_begin(self, args, state, control, model=None, **kwargs):\n            self.initial_signature = self.signature(model)\n\n        def on_step_end(self, args, state, control, **kwargs):\n            now = clock()\n            atomic_json(progress_path, {"optimizer_steps": state.global_step, "updated": now})\n            if now >= deadline:\n                control.should_training_stop = True\n                control.should_save = state.global_step > 0\n                control.should_evaluate = False\n            elif now - self.last_save >= checkpoint_seconds:\n                control.should_save = True\n            return control\n\n        def on_save(self, args, state, control, **kwargs):\n            self.last_save = clock()\n            signature = self.signature(kwargs["model"])\n            atomic_json(\n                Path(progress_path).parent / "training_integrity.json",\n                {\n                    "optimizer_steps": state.global_step,\n                    "adapter_updated": signature != self.initial_signature,\n                },\n            )\n\n    return DeadlineCallback()\n\n\ndef train_worker(config):\n    # Never use batch=1 without explicit negatives. Accumulation does not enlarge InfoNCE batches.\n    os.environ["INFONCE_USE_BATCH"] = "false"\n    os.environ["INFONCE_TEMPERATURE"] = "0.05"\n    os.environ["INFONCE_HARD_NEGATIVES"] = "1"\n    from swift.llm import sft_main\n    from swift.plugin import extra_callbacks\n\n    if time.time() >= config["train_deadline"]:\n        raise RuntimeError("Training deadline reached before loading model.")\n    extra_callbacks.append(\n        deadline_callback(config["train_deadline"], Path(config["run"]) / "progress.json")\n    )\n    result = sft_main(swift_arguments(config))\n    atomic_json(Path(config["run"]) / "train_result.json", result)\n\n\ndef run_bounded(command, log_path, deadline, env=None):\n    """A hung setup/evaluation cannot consume the entire 12h session."""\n    if time.time() >= deadline:\n        return {"returncode": None, "timed_out": True}\n    with Path(log_path).open("w", encoding="utf-8") as log:\n        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env)\n        last_notice = time.time()\n        try:\n            while process.poll() is None:\n                if time.time() - last_notice >= 60:\n                    remaining = max(0, deadline - time.time()) / 3600\n                    print(\n                        f"{Path(log_path).name}: running; budget remaining {remaining:.2f}h",\n                        flush=True,\n                    )\n                    last_notice = time.time()\n                if time.time() >= deadline:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=30)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait(timeout=15)\n                    return {"returncode": process.returncode, "timed_out": True}\n                time.sleep(2)\n            return {"returncode": process.returncode, "timed_out": False}\n        finally:\n            if process.poll() is None:\n                process.kill()\n                process.wait(timeout=15)\n\n\ndef complete_checkpoints(output_dir):\n    """Ignore incomplete writes after a hard timeout. Full checkpoints retain resume state."""\n    result = []\n    for path in Path(output_dir).rglob("checkpoint-*"):\n        required = (\n            "adapter_config.json",\n            "adapter_model.safetensors",\n            "trainer_state.json",\n            "optimizer.pt",\n            "scheduler.pt",\n            "rng_state.pth",\n        )\n        if not path.is_dir() or not all((path / name).is_file() for name in required):\n            continue\n        try:\n            state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))\n            if state["global_step"] > 0:\n                result.append((state["global_step"], path, state))\n        except (OSError, ValueError, KeyError):\n            continue\n    return sorted(result)\n\n\ndef export_adapter(config, process_status):\n    integrity_path = Path(config["run"]) / "training_integrity.json"\n    integrity = json.loads(integrity_path.read_text()) if integrity_path.exists() else {}\n    if not integrity.get("adapter_updated"):\n        raise RuntimeError("No verified adapter weight update; inspect training_integrity.json.")\n    checkpoints = [\n        item\n        for item in complete_checkpoints(config["output_dir"])\n        if item[0] <= integrity.get("optimizer_steps", 0)\n    ]\n    if not checkpoints:\n        raise RuntimeError(\n            "No complete trained checkpoint. Inspect train.log; no base model is exported."\n        )\n    _, latest, state = checkpoints[-1]\n    # State may record a best checkpoint from the original session path after resume.\n    best_name = Path(state.get("best_model_checkpoint") or "").name\n    chosen = next((item for item in checkpoints if item[1].name == best_name), checkpoints[-1])\n    step, selected, _ = chosen\n    output = Path(config["run"]) / "qwen3-adapter"\n    output.mkdir(exist_ok=True)\n    for name in ("adapter_model.safetensors", "adapter_config.json"):\n        shutil.copy2(selected / name, output / name)\n    manifest = {\n        "format": "swift_qwen3_adapter_v1",\n        "model_family": "qwen3",\n        "fine_tuned": True,\n        "optimizer_steps": step,\n        "base_model": config["base_model"],\n        "base_revision": config["base_revision"],\n        "query_instruction": config["instruction"],\n        "max_seq_length": config["max_length"],\n        "suffix": "<|endoftext|>",\n        "loss": "infonce",\n        "infonce": {"temperature": 0.05, "use_batch": False, "hard_negatives": 1},\n        "quantization": "bnb NF4 double quantization, FP16 compute",\n        "negative_policy": "same-paper lexical silver",\n        "selection": "best_eval_loss" if selected.name == best_name else "last_unvalidated",\n        "process_status": process_status,\n        "packages": config.get("packages", {}),\n        "data_sha256": config["data_sha256"],\n    }\n    atomic_json(output / "research_model.json", manifest)\n    # Package the last checkpoint too, including optimizer/RNG, separately from the deploy adapter.\n    resume = Path(config["run"]) / "resume"\n    if not resume.exists():\n        shutil.copytree(latest, resume)\n    atomic_json(\n        Path(config["run"]) / "resume_recipe.json",\n        {\n            "dataset_revision": config.get("dataset_revision", ""),\n            **{\n                key: config[key]\n                for key in (\n                    "base_model",\n                    "base_revision",\n                    "instruction",\n                    "max_length",\n                    "learning_rate",\n                    "data_sha256",\n                )\n            },\n        },\n    )\n    return output\n\n\ndef evaluate_worker(config):\n    import numpy as np\n\n\n    run = Path(config["run"])\n    panel = json.loads((run / "evaluation_panel.json").read_text(encoding="utf-8"))\n    encoder = QwenAdapterEncoder(\n        config["base_path"],\n        config["instruction"],\n        config["max_length"],\n        adapter_path=run / "qwen3-adapter",\n    )\n\n    def measure():\n        vectors = []\n        for text in panel["documents"]:\n            if time.time() >= config["eval_deadline"]:\n                raise TimeoutError("Evaluation reserve reached; no partial metrics reported.")\n            vectors.append(encoder.encode_document([text], batch_size=1)[0])\n        corpus = np.asarray(vectors)\n        recall, reciprocal = [], []\n        for query, gold in zip(panel["questions"], panel["gold"], strict=True):\n            if time.time() >= config["eval_deadline"]:\n                raise TimeoutError("Evaluation reserve reached; no partial metrics reported.")\n            query_vector = encoder.encode_query([query], batch_size=1)[0]\n            ranking = np.argsort(-(corpus @ query_vector)).tolist()\n            recall.append(len(set(ranking[:10]) & set(gold)) / len(gold))\n            reciprocal.append(\n                next((1 / (i + 1) for i, item in enumerate(ranking[:10]) if item in gold), 0)\n            )\n        return {"recall@10": float(np.mean(recall)), "mrr@10": float(np.mean(reciprocal))}\n\n    with encoder.model.disable_adapter():\n        before = measure()\n    after = measure()\n    atomic_json(\n        run / "retrieval_validation.json",\n        {\n            "baseline": before,\n            "fine_tuned": after,\n            "queries": len(panel["questions"]),\n            "corpus": len(panel["documents"]),\n            "scope": "fixed validation subset, not full benchmark",\n            "quantization": "same NF4 base for both evaluations",\n            "test_set_used": False,\n        },\n    )\n\n"""Inference for the SWIFT Qwen3 adapter recipe; identical query/EOS policy to training."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef adapter_identity(path, max_seq_length):\n    root = Path(path)\n    digest = hashlib.sha256(f"swift-qwen3-eos-v1:{max_seq_length}".encode())\n    for name in ("research_model.json", "adapter_config.json", "adapter_model.safetensors"):\n        digest.update(name.encode())\n        with (root / name).open("rb") as stream:\n            while block := stream.read(1024 * 1024):\n                digest.update(block)\n    return "research-qwen3-" + digest.hexdigest()[:32]\n\n\ndef read_adapter_manifest(path):\n    root = Path(path)\n    manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))\n    if (\n        manifest.get("format") != "swift_qwen3_adapter_v1"\n        or manifest.get("model_family") != "qwen3"\n        or manifest.get("fine_tuned") is not True\n        or manifest.get("optimizer_steps", 0) < 1\n        or manifest.get("base_model")\n        not in {"Qwen/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-0.6B"}\n        or len(manifest.get("base_revision", "")) != 40\n        or not manifest.get("query_instruction")\n        or not (root / "adapter_model.safetensors").is_file()\n    ):\n        raise ValueError("Invalid trained Qwen3 adapter manifest; base fallback is forbidden.")\n    return manifest\n\n\ndef extract_adapter_archive(archive_path, destination):\n    import zipfile\n\n    root = Path(destination).resolve()\n    if root.exists() and any(root.iterdir()):\n        raise ValueError("Choose an empty adapter extraction directory.")\n    with zipfile.ZipFile(archive_path) as archive:\n        size = 0\n        for item in archive.infolist():\n            target = (root / item.filename.replace("\\\\", "/")).resolve()\n            if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("Unsafe adapter archive entry.")\n            size += item.file_size\n            if size > 2 * 1024**3:\n                raise ValueError("Adapter archive exceeds 2 GiB.")\n        archive.extractall(root)\n    read_adapter_manifest(root)\n    return root\n\n\nclass QwenAdapterEncoder:\n    """Load a pinned base plus optional adapter. Use base-only solely for baseline evaluation."""\n\n    def __init__(self, base_path, instruction, max_seq_length=512, adapter_path=None):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        self.max_seq_length, self.instruction = max_seq_length, instruction\n        self.tokenizer = AutoTokenizer.from_pretrained(str(base_path), trust_remote_code=False)\n        self.tokenizer.padding_side = "right"\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.suffix = self.tokenizer.convert_tokens_to_ids("<|endoftext|>")\n        if self.suffix is None or self.suffix == self.tokenizer.unk_token_id:\n            raise ValueError("Qwen tokenizer must contain the SWIFT endoftext suffix.")\n        model = AutoModelForCausalLM.from_pretrained(\n            str(base_path),\n            trust_remote_code=False,\n            local_files_only=True,\n            torch_dtype=torch.float16,\n            attn_implementation="sdpa",\n            device_map={"": 0},\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type="nf4",\n                bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n            ),\n        )\n        if adapter_path:\n            from peft import PeftModel\n\n            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)\n        self.model = model.eval()\n\n    def get_sentence_embedding_dimension(self):\n        return int(self.model.config.hidden_size)\n\n    def encode_query(self, texts, **kwargs):\n        return self._encode(\n            [f"Instruct: {self.instruction}\\nQuery:{text}" for text in texts], **kwargs\n        )\n\n    def encode_document(self, texts, **kwargs):\n        return self._encode(texts, **kwargs)\n\n    def _encode(self, texts, batch_size=2, **kwargs):\n        import numpy as np\n        import torch\n        import torch.nn.functional as functional\n\n        result = []\n        with torch.inference_mode():\n            for start in range(0, len(texts), batch_size):\n                ids = self.tokenizer(\n                    texts[start : start + batch_size],\n                    add_special_tokens=False,\n                    truncation=True,\n                    max_length=self.max_seq_length - 1,\n                )["input_ids"]\n                batch = self.tokenizer.pad(\n                    {"input_ids": [row + [self.suffix] for row in ids]},\n                    padding=True,\n                    return_tensors="pt",\n                ).to(self.model.device)\n                base = (\n                    self.model.get_base_model()\n                    if hasattr(self.model, "peft_config")\n                    else self.model\n                )\n                # Use the transformer with adapters; avoid the vocabulary-sized LM head.\n                hidden = base.model(**batch, use_cache=False, return_dict=True).last_hidden_state\n                last = batch["attention_mask"].sum(dim=1) - 1\n                vectors = hidden[torch.arange(len(last), device=hidden.device), last].float()\n                result.append(functional.normalize(vectors, dim=1).cpu().numpy())\n        return (\n            np.concatenate(result)\n            if result\n            else np.empty((0, self.get_sentence_embedding_dimension()))\n        )\n\n\ndef load_swift_qwen_adapter(path, max_seq_length=512):\n    from huggingface_hub import snapshot_download\n\n    manifest = read_adapter_manifest(path)\n    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])\n    encoder = QwenAdapterEncoder(\n        base, manifest["query_instruction"], max_seq_length, adapter_path=path\n    )\n    return encoder, adapter_identity(path, max_seq_length)\n'
QWEN_WORKER_SOURCE = QWEN_LIBRARY_SOURCE + '\nif __name__ == "__main__":\n    import sys\n    config = json.loads(Path(sys.argv[2]).read_text(encoding="utf-8"))\n    action = {"prepare": prepare_worker, "train": train_worker, "evaluate": evaluate_worker}\n    action[sys.argv[1]](config)\n'
QWEN_ADAPTER_SOURCE = '"""Inference for the SWIFT Qwen3 adapter recipe; identical query/EOS policy to training."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef adapter_identity(path, max_seq_length):\n    root = Path(path)\n    digest = hashlib.sha256(f"swift-qwen3-eos-v1:{max_seq_length}".encode())\n    for name in ("research_model.json", "adapter_config.json", "adapter_model.safetensors"):\n        digest.update(name.encode())\n        with (root / name).open("rb") as stream:\n            while block := stream.read(1024 * 1024):\n                digest.update(block)\n    return "research-qwen3-" + digest.hexdigest()[:32]\n\n\ndef read_adapter_manifest(path):\n    root = Path(path)\n    manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))\n    if (\n        manifest.get("format") != "swift_qwen3_adapter_v1"\n        or manifest.get("model_family") != "qwen3"\n        or manifest.get("fine_tuned") is not True\n        or manifest.get("optimizer_steps", 0) < 1\n        or manifest.get("base_model")\n        not in {"Qwen/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-0.6B"}\n        or len(manifest.get("base_revision", "")) != 40\n        or not manifest.get("query_instruction")\n        or not (root / "adapter_model.safetensors").is_file()\n    ):\n        raise ValueError("Invalid trained Qwen3 adapter manifest; base fallback is forbidden.")\n    return manifest\n\n\ndef extract_adapter_archive(archive_path, destination):\n    import zipfile\n\n    root = Path(destination).resolve()\n    if root.exists() and any(root.iterdir()):\n        raise ValueError("Choose an empty adapter extraction directory.")\n    with zipfile.ZipFile(archive_path) as archive:\n        size = 0\n        for item in archive.infolist():\n            target = (root / item.filename.replace("\\\\", "/")).resolve()\n            if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("Unsafe adapter archive entry.")\n            size += item.file_size\n            if size > 2 * 1024**3:\n                raise ValueError("Adapter archive exceeds 2 GiB.")\n        archive.extractall(root)\n    read_adapter_manifest(root)\n    return root\n\n\nclass QwenAdapterEncoder:\n    """Load a pinned base plus optional adapter. Use base-only solely for baseline evaluation."""\n\n    def __init__(self, base_path, instruction, max_seq_length=512, adapter_path=None):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n        self.max_seq_length, self.instruction = max_seq_length, instruction\n        self.tokenizer = AutoTokenizer.from_pretrained(str(base_path), trust_remote_code=False)\n        self.tokenizer.padding_side = "right"\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.suffix = self.tokenizer.convert_tokens_to_ids("<|endoftext|>")\n        if self.suffix is None or self.suffix == self.tokenizer.unk_token_id:\n            raise ValueError("Qwen tokenizer must contain the SWIFT endoftext suffix.")\n        model = AutoModelForCausalLM.from_pretrained(\n            str(base_path),\n            trust_remote_code=False,\n            local_files_only=True,\n            torch_dtype=torch.float16,\n            attn_implementation="sdpa",\n            device_map={"": 0},\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type="nf4",\n                bnb_4bit_use_double_quant=True,\n                bnb_4bit_compute_dtype=torch.float16,\n            ),\n        )\n        if adapter_path:\n            from peft import PeftModel\n\n            model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)\n        self.model = model.eval()\n\n    def get_sentence_embedding_dimension(self):\n        return int(self.model.config.hidden_size)\n\n    def encode_query(self, texts, **kwargs):\n        return self._encode(\n            [f"Instruct: {self.instruction}\\nQuery:{text}" for text in texts], **kwargs\n        )\n\n    def encode_document(self, texts, **kwargs):\n        return self._encode(texts, **kwargs)\n\n    def _encode(self, texts, batch_size=2, **kwargs):\n        import numpy as np\n        import torch\n        import torch.nn.functional as functional\n\n        result = []\n        with torch.inference_mode():\n            for start in range(0, len(texts), batch_size):\n                ids = self.tokenizer(\n                    texts[start : start + batch_size],\n                    add_special_tokens=False,\n                    truncation=True,\n                    max_length=self.max_seq_length - 1,\n                )["input_ids"]\n                batch = self.tokenizer.pad(\n                    {"input_ids": [row + [self.suffix] for row in ids]},\n                    padding=True,\n                    return_tensors="pt",\n                ).to(self.model.device)\n                base = (\n                    self.model.get_base_model()\n                    if hasattr(self.model, "peft_config")\n                    else self.model\n                )\n                # Use the transformer with adapters; avoid the vocabulary-sized LM head.\n                hidden = base.model(**batch, use_cache=False, return_dict=True).last_hidden_state\n                last = batch["attention_mask"].sum(dim=1) - 1\n                vectors = hidden[torch.arange(len(last), device=hidden.device), last].float()\n                result.append(functional.normalize(vectors, dim=1).cpu().numpy())\n        return (\n            np.concatenate(result)\n            if result\n            else np.empty((0, self.get_sentence_embedding_dimension()))\n        )\n\n\ndef load_swift_qwen_adapter(path, max_seq_length=512):\n    from huggingface_hub import snapshot_download\n\n    manifest = read_adapter_manifest(path)\n    base = snapshot_download(manifest["base_model"], revision=manifest["base_revision"])\n    encoder = QwenAdapterEncoder(\n        base, manifest["query_instruction"], max_seq_length, adapter_path=path\n    )\n    return encoder, adapter_identity(path, max_seq_length)\n'
exec(QWEN_LIBRARY_SOURCE, globals())

In [ ]:
# A private environment avoids changing Kaggle's already-loaded native libraries.
# Keep Kaggle's CUDA-enabled PyTorch via system-site-packages and an exact constraint.
SESSION_STARTED = session_start(
    "/kaggle/temp/qwen-session-clock.json", SESSION_ALREADY_ELAPSED_HOURS)
TRAIN_DEADLINE = SESSION_STARTED + 10 * 3600
TRAIN_HARD_DEADLINE = SESSION_STARTED + 10.5 * 3600
EVAL_DEADLINE = SESSION_STARTED + 11 * 3600
RUN = Path("/kaggle/working") / RUN_NAME
assert not RUN.exists(), "Choose a new RUN_NAME; never overwrite a previous run."
RUN.mkdir(parents=True)
ENV = Path("/kaggle/temp/qwen-training-env")
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(ENV)
PYTHON = ENV / "bin/python"
constraints = ENV / "torch-constraints.txt"
torch_version = importlib.metadata.version("torch")
constraints.write_text("torch==" + torch_version + "\n", encoding="utf-8")
install_qwen_environment(PYTHON, QWEN_REQUIREMENTS, constraints, TRAIN_DEADLINE)
WORKER = Path("/kaggle/temp/qwen_worker.py")
WORKER.write_text(QWEN_WORKER_SOURCE, encoding="utf-8")
ADAPTER_MODULE = Path("/kaggle/temp/qwen_adapter_encoder.py")
ADAPTER_MODULE.write_text(QWEN_ADAPTER_SOURCE, encoding="utf-8")
preflight = r"""
import importlib.metadata as m, torch, numpy, pandas, datasets, bitsandbytes
from swift.llm import TrainArguments, sft_main
from swift.plugin import extra_callbacks
from transformers import Qwen3Model
from peft import PeftModel
from packaging.version import Version
assert torch.cuda.is_available(), "Enable Kaggle T4 GPU before Save Version"
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), (
    "Use a Kaggle image with PyTorch >=2.6 for safe optimizer-state resume")
assert torch.cuda.get_device_properties(0).total_memory >= 14 * 2**30
print("GPU:", torch.cuda.get_device_name(0))
print({p:m.version(p) for p in ["torch","ms-swift","transformers","peft","bitsandbytes"]})
"""
subprocess.run([str(PYTHON), "-c", preflight], check=True, timeout=180)
print("Session elapsed hours:", round((time.time() - SESSION_STARTED) / 3600, 2))

## Dữ liệu

Mặc định tải QASPER chính thức, giữ split theo paper; không gọi LLM tạo dữ liệu.
QASPER chủ yếu tiếng Anh, nên kết quả này chưa chứng minh chất lượng tiếng Việt.
Corpus riêng dùng cùng định dạng prepared-data của notebook E5. Một negative
được chọn theo overlap từ trong cùng paper, loại mọi gold đã biết; các negative
này là **silver**, vẫn có thể chứa đoạn liên quan chưa được gán nhãn. Các mẫu quá
dài bị loại trước khi train để không cắt mất gold. Test split không dùng chọn model.

Batch 1 có một positive + một negative. Gradient accumulation 8 chỉ tăng số mẫu
giữa hai lần cập nhật optimizer; không tăng số negative của một phép InfoNCE.
Validation loss chọn checkpoint. Có thêm phép đo retrieval trên một panel cố định
16 query / tối đa khoảng 256 đoạn validation, cùng quantization trước/sau; đó là
kiểm tra nhỏ, không đại diện toàn bộ benchmark.

In [ ]:
config = {
    "run": str(RUN), "base_model": BASE_MODEL, "base_revision": "",
    "max_length": MAX_LENGTH, "learning_rate": LEARNING_RATE,
    "max_steps": MAX_STEPS, "pair_limit": PAIR_LIMIT, "instruction": INSTRUCTION,
    "prepared_data": PREPARED_DATA, "resume": RESUME_CHECKPOINT,
    "output_dir": str(RUN / "checkpoints"),
    "train_deadline": TRAIN_DEADLINE, "eval_deadline": EVAL_DEADLINE,
}
assert BASE_MODEL in {"Qwen/Qwen3-Embedding-4B", "Qwen/Qwen3-Embedding-0.6B"}
assert 80 <= MAX_LENGTH <= 512 and MAX_STEPS > 0 and PAIR_LIMIT >= 16
atomic_json(RUN / "config.json", config)
status = run_bounded([str(PYTHON), str(WORKER), "prepare", str(RUN / "config.json")],
    RUN / "prepare.log", min(TRAIN_DEADLINE - 1800, time.time() + 3600))
if status["returncode"] != 0:
    print((RUN / "prepare.log").read_text(encoding="utf-8")[-8000:])
    raise RuntimeError("Preparation failed or timed out. Inspect prepare.log in Output.")
config = json.loads((RUN / "config.json").read_text(encoding="utf-8"))
print(json.loads((RUN / "triplet_statistics.json").read_text(encoding="utf-8")))
print("Pinned base revision:", config["base_revision"])
print("Train remaining hours:", round((TRAIN_DEADLINE - time.time()) / 3600, 2))

In [ ]:
# Fresh child process owns GPU memory and can be drained before evaluation/export.
print("Training; logs and progress are written continuously under", RUN)
train_status = run_bounded([str(PYTHON), str(WORKER), "train", str(RUN / "config.json")],
    RUN / "train.log", TRAIN_HARD_DEADLINE)
atomic_json(RUN / "process_status.json", train_status)
if train_status["returncode"] != 0:
    print((RUN / "train.log").read_text(encoding="utf-8")[-8000:])
    print("Attempting to preserve the last complete checkpoint.")
MODEL_OUT = export_adapter(config, train_status)
# Secure the small deploy artifact before any optional evaluation.
shutil.make_archive(str(RUN / "qwen3-adapter"), "zip", MODEL_OUT)
print("Trained adapter saved:", MODEL_OUT)
print(json.loads((MODEL_OUT / "research_model.json").read_text(encoding="utf-8")))

In [ ]:
if time.time() < EVAL_DEADLINE - 300:
    evaluation_status = run_bounded(
        [str(PYTHON), str(WORKER), "evaluate", str(RUN / "config.json")],
        RUN / "evaluate.log", EVAL_DEADLINE)
    atomic_json(RUN / "evaluation_status.json", evaluation_status)
    result_path = RUN / "retrieval_validation.json"
    if result_path.is_file():
        print(json.loads(result_path.read_text(encoding="utf-8")))
    else:
        print("Evaluation incomplete/skipped; no partial metric is reported. See evaluate.log.")
else:
    atomic_json(RUN / "evaluation_status.json", {"skipped": "export reserve"})
    print("Evaluation skipped to preserve output time.")

In [ ]:
# Keep deploy adapter and resume checkpoint separate. No 8GB merge is required.
from IPython.display import display, FileLink
for file in [RUN / "qwen3-adapter.zip", RUN / "config.json"]:
    display(FileLink(str(file)))
atomic_json(RUN / "session_report.json", {
    "session_limit_hours": 12, "train_stop_hours": 10, "train_hard_stop_hours": 10.5,
    "eval_stop_hours": 11, "elapsed_hours": (time.time() - SESSION_STARTED) / 3600,
    "optimizer_steps": read_adapter_manifest(MODEL_OUT)["optimizer_steps"],
    "resume_checkpoint": str(RUN / "resume"),
})
print("Save Version Output contains the adapter ZIP, logs, data, metrics and resume checkpoint.")
print("Next session: add this Output as Input; set RESUME_CHECKPOINT to its resume directory.")
print("To serve: add adapter ZIP as Input, open notebook 05 and set ADAPTER_ZIP.")
print("Use Qwen3 family in the app; changing checkpoint requires rebuilding document vectors.")